***
# Customer_Segmentation_Application
***

In [5]:
# 1. Import libraries

import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import os
import customer_scoring as cs

# 2. LOAD EXCEL DATABASE

test_file = "new_customers_test.xlsx"
if os.path.exists(test_file):
    new_customers_test = pd.read_excel(test_file)
else:
    print("Test database not found. Please check the file location.")
    
# 3. CREATE MAIN WINDOW

window = tk.Tk()
window.title("Customer Segmentation Application")
window.geometry("700x750")
window.configure(bg="#F0F4F8")

# 4. DEFINE COMMON FUNCTIONS

def validate_customer(form):

    # 1. Get customer information from the form
    fields = {
        "Customer ID": form["Customer ID"].get(),
        "Age": form["Age"].get(),
        "Gender": form["Gender"].get(),
        "Income": form["Income"].get(),
        "Spending Score": form["Spending Score"].get(),
        "Membership Years": form["Membership Years"].get(),
        "Purchase Frequency": form["Purchase Frequency"].get(),
        "Preferred Category": form["Preferred Category"].get(),
        "Last Purchase Amount": form["Last Purchase Amount"].get()
    }

    # 2. Check required fields
    for field_name, value in fields.items():

        if str(value).strip() == "":
            messagebox.showwarning(
                "Missing Information",
                f"Please enter {field_name}."
            )
            return None

    # 3. Convert numerical fields
    # 3. Convert numerical fields
    try:
        field_name = "Age"
        age = int(fields[field_name])
    
        field_name = "Income"
        income = float(fields[field_name])
    
        field_name = "Spending Score"
        spending_score = float(fields[field_name])
    
        field_name = "Membership Years"
        membership_years = int(fields[field_name])
    
        field_name = "Purchase Frequency"
        purchase_frequency = int(fields[field_name])
    
        field_name = "Last Purchase Amount"
        last_purchase_amount = float(fields[field_name])

    except (ValueError, TypeError):
        messagebox.showwarning(
            "Invalid Information",
            f"Please enter a valid numerical value for {field_name}."
        )
        return None
        
    # 4. Validate numerical ranges

    if not 18 <= age <=100 :
        messagebox.showwarning("Invalid Age", "Age must be between 18 and 100.")
        return None
    if income < 0:
        messagebox.showwarning("Invalid Income", "Income cannot be negative.")
        return None
    if not 1 <= spending_score <= 100:
        messagebox.showwarning("Invalid Spending Score", "Spending Score must be between 1 and 100.")
        return None

    if not 0 <= membership_years <= 25:
        messagebox.showwarning("Invalid Membership Years", "Membership Years must be between 0 and 25.")
        return None

    if purchase_frequency < 0 :
        messagebox.showwarning("Invalid Purchase Frequency", "Purchase Frequency cannot be negative.")
        return None

    if last_purchase_amount < 0.01:
        messagebox.showwarning("Invalid Purchase Amount", "Last Purchase Amount must be at least 0.01.")
        return None
        

        # 5. Return validated customer information using Excel column names
        
    return {
        "id": fields["Customer ID"].strip(),
        "age": age,
        "gender": fields["Gender"],
        "income": income,
        "spending_score": spending_score,
        "membership_years": membership_years,
        "purchase_frequency": purchase_frequency,
        "preferred_category": fields["Preferred Category"],
        "last_purchase_amount": last_purchase_amount
    }
    
def calculate_customer_segment(purchase_frequency, spending_score, last_purchase_amount):

    # 1. Normalize customer indicators
    pf_normalized = cs.normalize(purchase_frequency, 1, 50)
    ss_normalized = cs.normalize(spending_score, 1, 100)
    lpa_normalized = cs.normalize(last_purchase_amount, 10.40, 999.74)

    # 2. Calculate Customer Value Score
    cvs = cs.calculate_cvs(pf_normalized, ss_normalized, lpa_normalized)

    # 3. Assign customer segment
    segment = cs.assign_segment(cvs)
    
    return cvs, segment
    
def clear_form():

    # 1. Clear all registration fields
    for field in registration_form.values():
        if isinstance(field, ttk.Combobox):
            field.set("")
        else:
            field.delete(0, tk.END)  
            
def exit_application():
    messagebox.showinfo(
        "Goodbye!",
        "Thank you for using the Customer Segmentation Application!"
    )
    window.destroy()            
# 5. REGISTRATION FUNCTIONS

def submit_customer():

    # 1. Get Customer ID
    customer_id = registration_form["Customer ID"].get().strip()

    # 2. Check if Customer ID is empty
    if not customer_id:
        messagebox.showwarning(
            "Missing Information",
            "Please enter Customer ID."
        )
        return

    # 3. Check if Customer ID already exists
    if (new_customers_test["id"].astype(str).str.strip() == customer_id).any():
        messagebox.showwarning(
            "Duplicate Customer",
            "This Customer ID already exists. Please use Update Customer to update the customer's existing informations."
        )
        return

    # 4. Validate the remaining customer information
    validated_data = validate_customer(registration_form)

    if validated_data is None:
        return
        
    # 5. Calculate CVS and assign customer segment
    
    cvs, segment = calculate_customer_segment(
        validated_data["purchase_frequency"],
        validated_data["spending_score"],
        validated_data["last_purchase_amount"]
    )

    # 6. Add CVS and customer segment to the validated customer information
    
    validated_data["CVS"] = cvs
    validated_data["CVS_segment"] = segment
    
    # 7. Add the validated customer as a new row in the DataFrame
    
    new_customers_test.loc[len(new_customers_test)] = [
        validated_data[column_name] for column_name in new_customers_test.columns
    ]
    # 8. Save the updated DataFrame to the Excel file
    
    new_customers_test.to_excel(test_file, index=False)

    # 9. Display a confirmation message after successful registration
    
    messagebox.showinfo(
        "Customer Registration Successful",
        "The customer has been successfully registered!\n\n"
        f"Customer ID: {validated_data['id']}\n"
        f"CVS: {cvs:.2f}\n"
        f"Customer Segment: {segment}"
    )
    # 10. Clear the registration form after successful registration
    clear_form()
    
# 6. Define the Update Customer Function

def update_customer():

    # 1. Create the Update window
    update_window = tk.Toplevel(window)
    update_window.title("Update Existing Customer")
    update_window.geometry("400x250")
    update_window.configure(bg="#F0F4F8")
    update_window.transient(window)
    
    # 2. Create the Customer ID search field
    tk.Label(
    update_window,
    text="Enter Customer ID:",
    bg="#F0F4F8",
    fg="#1E3A5F",
    font=("Arial", 11, "bold")
    ).pack(pady=10)
    search_id_entry = tk.Entry(update_window)
    search_id_entry.pack(pady=5)
    
    # 3. Define the Search Customer function
    def search_customer():

        # 3.1. Get the Customer ID
        customer_id = search_id_entry.get().strip()

        # 3.2. Check if Customer ID is empty
        if not customer_id:
            messagebox.showwarning(
                "Missing Information",
                "Please enter Customer ID."
            )
            return

        # 3.3. Check if Customer ID exists
        if not (new_customers_test["id"].astype(str).str.strip() == customer_id).any():
            messagebox.showwarning(
                "Customer Not Found",
                "This Customer ID does not exist."
            )
            return
        # 3.4. Retrieve the existing customer's information
        customer_data = new_customers_test[
            new_customers_test["id"].astype(str).str.strip() == customer_id
        ].iloc[0]
        # 3.5. Display a confirmation message when the customer is found
        messagebox.showinfo(
            "Customer Found",
            f"Customer ID: {customer_id} was found successfully.\n"
            "You can now update the customer information."
        )
        # 3.6. Create a separate window for editing customer information
        
        edit_window = tk.Toplevel(window)
        edit_window.title("Edit Customer Information")
        edit_window.geometry("600x750")
        edit_window.transient(window)
        edit_window.configure(bg="#F0F4F8")
        update_window.withdraw()
        
        # 3.7. Customer ID (Read-only)
        tk.Label(edit_window, text="Customer ID:").pack(pady=5)

        update_id_entry = tk.Entry(edit_window)
        update_id_entry.pack(pady=5)

        update_id_entry.insert(0, customer_id)
        update_id_entry.config(state="readonly")
        
        # 3.7. Display the customer's existing Age
        tk.Label(edit_window, text="Age:").pack(pady=5)

        update_age_entry = tk.Entry(edit_window)
        update_age_entry.pack(pady=5)

        update_age_entry.insert(0, str(customer_data["age"]))
        
        # 3.8. Gender
        tk.Label(edit_window, text="Gender:").pack(pady=5)

        update_gender_combobox = ttk.Combobox(
        edit_window,
        values=["Male", "Female", "Other"],
        state="readonly"
            )
        update_gender_combobox.pack(pady=5)
        update_gender_combobox.set(customer_data["gender"])

        # 3.9. Income
        tk.Label(edit_window, text="Income:").pack(pady=5)

        update_income_entry = tk.Entry(edit_window)
        update_income_entry.pack(pady=5)
        update_income_entry.insert(0, str(customer_data["income"]))


        # 3.10. Spending Score
        tk.Label(edit_window, text="Spending Score:").pack(pady=5)

        update_spending_score_entry = tk.Entry(edit_window)
        update_spending_score_entry.pack(pady=5)
        update_spending_score_entry.insert(0, str(customer_data["spending_score"]))


        # 3.11. Membership Years
        tk.Label(edit_window, text="Membership Years:").pack(pady=5)

        update_membership_years_entry = tk.Entry(edit_window)
        update_membership_years_entry.pack(pady=5)
        update_membership_years_entry.insert(0, str(customer_data["membership_years"]))


        # 3.12. Purchase Frequency
        tk.Label(edit_window, text="Purchase Frequency:").pack(pady=5)

        update_purchase_frequency_entry = tk.Entry(edit_window)
        update_purchase_frequency_entry.pack(pady=5)
        update_purchase_frequency_entry.insert(0, str(customer_data["purchase_frequency"]))


        # 3.13. Preferred Category
        tk.Label(edit_window, text="Preferred Category:").pack(pady=5)

        update_preferred_category_combobox = ttk.Combobox(
            edit_window,
            values=["Electronics", "Clothing", "Groceries", "Home & Garden", "Sports"],
            state="readonly"
        )
        update_preferred_category_combobox.pack(pady=5)
        update_preferred_category_combobox.set(customer_data["preferred_category"])


        # 3.14. Last Purchase Amount
        tk.Label(edit_window, text="Last Purchase Amount:").pack(pady=5)

        update_last_purchase_amount_entry = tk.Entry(edit_window)
        update_last_purchase_amount_entry.pack(pady=5)
        update_last_purchase_amount_entry.insert(0, str(customer_data["last_purchase_amount"]))

        # 3.15. Store customer information fields in a dictionary
        update_form = {
            "Customer ID": search_id_entry,
            "Age": update_age_entry,
            "Gender": update_gender_combobox,
            "Income": update_income_entry,
            "Spending Score": update_spending_score_entry,
            "Membership Years": update_membership_years_entry,
            "Purchase Frequency": update_purchase_frequency_entry,
            "Preferred Category": update_preferred_category_combobox,
            "Last Purchase Amount": update_last_purchase_amount_entry
        }
        # 3.15. Define the Save Changes function
        def save_changes():

            # 1. Validate the updated customer information
            validated_data = validate_customer(update_form)
        
            if validated_data is None:
                return
        
            # 2. Ask for confirmation before updating
            confirm_update = messagebox.askyesno(
                "Confirm Customer Update",
                f"Are you sure you want to update Customer ID {customer_id}?\n\n"
                "The existing customer information will be modified."
            )
        
            if not confirm_update:
                return
        
            # 3. Recalculate CVS and assign customer segment
            cvs, segment = calculate_customer_segment(
                validated_data["purchase_frequency"],
                validated_data["spending_score"],
                validated_data["last_purchase_amount"]
            )
        
            # 4. Add the updated CVS and segment to customer information
            validated_data["CVS"] = cvs
            validated_data["CVS_segment"] = segment
        
            # 5. Find the existing customer in the DataFrame
            customer_index = new_customers_test[
                new_customers_test["id"].astype(str).str.strip() == customer_id
            ].index[0]  # The index label of the first matching row
        
            # 6. Update the existing customer's information in the DataFrame
            for column_name in new_customers_test.columns:
                if column_name != "id":
                    new_customers_test.loc[customer_index, column_name] = validated_data[column_name]
        
            # 7. Save the updated DataFrame to the Excel file
            new_customers_test.to_excel(test_file, index=False)
        
            # 8. Display a confirmation message after successful update
                            
            messagebox.showinfo(
                    "Customer Updated Successfully",
                    "The customer's information has been successfully updated!\n\n"
                    f"Customer ID: {customer_id}\n"
                    f"CVS: {cvs:.2f}\n"
                    f"Customer Segment: {segment}"
                            )
            edit_window.destroy()
            
        # 3.16. Create the Save Changes button
        save_button = tk.Button(
            edit_window,
            text="Save Changes",
            command=save_changes,
            bg="#2E7D32",
            fg="white",
            activebackground="#1B5E20",
            activeforeground="white",
            font=("Arial", 10, "bold"),
            width=15,
            height=1)
        save_button.pack(pady=10)
        
    # 4. Create the Search Customer button
    
    search_button = tk.Button(
        update_window,
        text="Search Customer",
        command=search_customer,
        bg="#1976D2",
        fg="white",
        activebackground="#0D47A1",
        activeforeground="white",
        font=("Arial", 10, "bold"),
        width=15,
        height=1
                )
    search_button.pack(pady=15)
    
# 7. Create the main registration form

# Registration form title
tk.Label(
    window,
    text="Customer Registration",
    font=("Arial", 16, "bold")
).pack(pady=15)


# 7.1. Customer ID
tk.Label(window, text="Customer ID:").pack(pady=5)
id_entry = tk.Entry(window)
id_entry.pack(pady=5)


# 7.2. Age
tk.Label(window, text="Age:").pack(pady=5)
age_entry = tk.Entry(window)
age_entry.pack(pady=5)


# 7.3. Gender
tk.Label(window, text="Gender:").pack(pady=5)
gender_combobox = ttk.Combobox(
    window,
    values=["Male", "Female", "Other"],
    state="readonly"
)
gender_combobox.pack(pady=5)


# 7.4. Income
tk.Label(window, text="Income:").pack(pady=5)
income_entry = tk.Entry(window)
income_entry.pack(pady=5)


# 7.5. Spending Score
tk.Label(window, text="Spending Score:").pack(pady=5)
spending_score_entry = tk.Entry(window)
spending_score_entry.pack(pady=5)


# 7.6. Membership Years
tk.Label(window, text="Membership Years:").pack(pady=5)
membership_years_entry = tk.Entry(window)
membership_years_entry.pack(pady=5)


# 7.7. Purchase Frequency
tk.Label(window, text="Purchase Frequency:").pack(pady=5)
purchase_frequency_entry = tk.Entry(window)
purchase_frequency_entry.pack(pady=5)


# 7.8. Preferred Category
tk.Label(window, text="Preferred Category:").pack(pady=5)
preferred_category_combobox = ttk.Combobox(
    window,
    values=["Electronics", "Clothing", "Groceries", "Home & Garden", "Sports"],
    state="readonly"
)
preferred_category_combobox.pack(pady=5)


# 7.9. Last Purchase Amount
tk.Label(window, text="Last Purchase Amount:").pack(pady=5)
last_purchase_amount_entry = tk.Entry(window)
last_purchase_amount_entry.pack(pady=5)


# 7.10. Store registration fields in a dictionary
registration_form = {
    "Customer ID": id_entry,
    "Age": age_entry,
    "Gender": gender_combobox,
    "Income": income_entry,
    "Spending Score": spending_score_entry,
    "Membership Years": membership_years_entry,
    "Purchase Frequency": purchase_frequency_entry,
    "Preferred Category": preferred_category_combobox,
    "Last Purchase Amount": last_purchase_amount_entry
}

# 8. Create the main application buttons

# 8.1. Create a frame to organize the buttons
button_frame = tk.Frame(window)
button_frame.pack(pady=20)

# 8.2. Submit Customer button
submit_button = tk.Button(
    button_frame,
    text="Submit Customer",
    command=submit_customer,
    bg="#2E7D32",
    fg="white",
    activebackground="#1B5E20",
    activeforeground="white",
    font=("Arial", 10, "bold"),
    width=15,
    height=1
)
submit_button.pack(side="left", padx=10)

# 8.3. Update Existing Customer button
update_button = tk.Button(
    button_frame,
    text="Update Customer",
    command=update_customer,
    bg="#1976D2",
    fg="white",
    activebackground="#0D47A1",
    activeforeground="white",
    font=("Arial", 10, "bold"),
    width=15,
    height=1
)
update_button.pack(side="left", padx=10)
# 8.4. Exit button
exit_button = tk.Button(
    button_frame,
    text="Exit",
    command=exit_application,
    bg="#D32F2F",
    fg="white",
    activebackground="#B71C1C",
    activeforeground="white",
    font=("Arial", 10, "bold"),
    width=15,
    height=1
)
exit_button.pack(side="left", padx=10)
# 9. Run the application
window.mainloop()

In [4]:
new_customers_test.head(13)

,id,age,gender,income,spending_score,membership_years,purchase_frequency,preferred_category,last_purchase_amount,CVS,CVS_segment
0,1001,24,Female,2200,20,1,5,Clothing,75,11.294930,Lower CVS
1,1002,45,Female,12,58,12,78,Clothing,456,86.586247,Higher CVS
2,1003,51,Female,6800,92,15,45,Sports,850,88.859923,Higher CVS
3,1004,29,Male,3100,40,3,15,Electronics,230,30.053995,Lower CVS
4,1005,43,Female,5200,75,10,35,Home & Garden,650,69.594797,Higher CVS
5,1006,40,Male,500000,50,12,40,Home & Garden,1230,84.120296,Higher CVS
6,1007,45,Female,50000,45,4,41,Sports,450,56.836920,Medium CVS
7,1008,45,Female,45000,12,15,22,Electronics,4555,171.108332,Higher CVS
8,1009,15,Female,45000,12,4,5,Home & Garden,4500,157.690619,Higher CVS
9,1010,28,Female,6000000,90,4,20,Home & Garden,5000,211.003575,Higher CVS
